# Plots for AE Guidance website "Guidance on Using Climate Data for Decision Making" 
Author: Nicole Keeney<br> 
Date: June 2026 

In [ ]:
#!pip install dataframe-image

In [ ]:
import holoviews as hv
import hvplot.pandas
from bokeh.resources import CDN
from bokeh.embed import file_html
import pandas as pd
import numpy as np
from IPython.display import display
import dataframe_image as dfi

# Disable zoom / active scrolling
# Fixes the plot in place, better for website :) 
hv.plotting.bokeh.element.ElementPlot.active_tools = ['pan']
hv.extension("bokeh")

In [ ]:
def load_csv(path):
    return pd.read_csv(path, index_col="Year")

hist_data   = load_csv("data/tas_global_Historical.csv")
ssp119_data = load_csv("data/tas_global_SSP1_1_9.csv")
ssp126_data = load_csv("data/tas_global_SSP1_2_6.csv")
ssp245_data = load_csv("data/tas_global_SSP2_4_5.csv")
ssp370_data = load_csv("data/tas_global_SSP3_7_0.csv")
ssp585_data = load_csv("data/tas_global_SSP5_8_5.csv")

SSP_MAPPING = {
    "SSP 1-1.9": (ssp119_data, "#00a9cf", "SSP1-1.9"),
    "SSP 1-2.6": (ssp126_data, "#003466", "SSP1-2.6"),
    "SSP 2-4.5": (ssp245_data, "#f69320", "SSP2-4.5"),
    "SSP 3-7.0": (ssp370_data, "#df0000", "SSP3-7.0"),
    "SSP 5-8.5": (ssp585_data, "#980002", "SSP5-8.5"),
}

In [ ]:
def plot_warming_trajectories(warming_level=1.5, ssp=None, width=650, height=350):
    if ssp is None:
        ssp = "All"

    plot = hist_data.hvplot(
        x="Year", y="Mean", color="k", label="Historical", width=width, height=height,
    ) * hist_data.hvplot.area(
        x="Year", y="5%", y2="95%", alpha=0.1, color="k",
        ylabel="°C", xlabel="", ylim=[-1, 5], xlim=[1950, 2100],
        hover=False
    ).opts(tools=["pan", "wheel_zoom"])

    if ssp in SSP_MAPPING:
        data, color, label = SSP_MAPPING[ssp]

        plot *= data.hvplot.area(
            x="Year", y="5%", y2="95%", alpha=0.2, color=color, label="90% interval", 
            hover=False
        )
        plot *= data.hvplot(x="Year", y="Mean", color=color, label=label, line_width=2)

        year_crosses = data.index[data["Mean"] > warming_level]
        if len(year_crosses) > 0:
            yr = year_crosses[0]
            plot *= hv.VLine(yr).opts(color=color, line_dash="dashed", line_width=1, tools=[])
            plot *= hv.Text(yr + 6, 4.5, str(int(yr))).opts(
                text_font_size="11pt", color=color, tools=[]
            )

        years_95 = data.index[data["95%"] > warming_level]
        years_5  = data.index[data["5%"]  > warming_level]
        if len(years_95) > 0 and len(years_5) > 0:
            x_95, x_5 = years_95[0], years_5[0]
            plot *= hv.VLine(x_95).opts(color=color, line_width=1, tools=[])
            plot *= hv.VLine(x_5).opts(color=color, line_width=1, tools=[])
            yr_rng = x_5 - x_95
            if yr_rng > 0:
                plot *= hv.Curve([(x_95, -0.5), (x_5, -0.5)]).opts(color=color, line_width=1, tools=[])
                plot *= hv.Text(x_95 + yr_rng / 2, -0.25, f"{int(yr_rng)}yrs").opts(
                    text_font_size="11pt", color=color, tools=[]
                )

    elif ssp != "All":
        raise ValueError(f"Invalid ssp: {ssp!r}. Must be one of {list(SSP_MAPPING)} or 'All'.")

    else:
        for data, color, label in SSP_MAPPING.values():
            plot *= data.hvplot(x="Year", y="Mean", color=color, label=label)

    plot *= hv.HLine(warming_level).opts(color="black", line_width=1.0, tools=[])
    plot *= hv.Text(1970, warming_level + 0.4, f"{warming_level}°C warming level").opts(
        text_font_size="11pt", tools=[]
    )

    plot.opts(
        title="Global mean surface temperature change relative to 1850-1900",
        fontsize={"title": 12, "labels": 11, "ticks": 11, "legend": 11},
        legend_position="bottom",
        default_tools=[],
    )
    return plot

In [ ]:
p = plot_warming_trajectories(ssp="SSP 3-7.0", warming_level=0.8)
p

In [ ]:
with open("warming_trajectory.html", "w") as f:
    f.write(file_html(hv.render(p), CDN))

print("Saved warming_trajectory.html")

In [ ]:
import pandas as pd
import numpy as np
from IPython.display import display

print("Global Warming Levels and Approximate 30-Year Climatology Windows Center years and 30-year time ranges associated with each global  warming level, based on the IPCC AR6 multi-model ensemble mean  global mean surface temperature change relative to 1850–1900 under SSP 3-7.0.")

# Combine historical + SSP 3-7.0 into one continuous time series
combined = pd.concat([hist_data[["Mean"]], ssp370_data[["Mean"]]])

# Compute 30-year climatology window for each warming level
warming_levels = [0.8, 1.0, 1.2, 1.5, 2.0, 2.5, 3.0, 4.0]
rows = []
for wl in warming_levels:
    crosses = combined.index[combined["Mean"] > wl]
    if len(crosses) > 0:
        center = int(crosses[0])
        rows.append({
            "Global Warming Level": f"{wl} °C",
            "Approximate equivalent time range for 30-year climatology": f"{center-15} – {center+15}\n(center year {center})",
        })
    else:
        rows.append({
            "Global Warming Level": f"{wl} °C",
            "Approximate equivalent time range for 30-year climatology": "Not reached",
        })

df = pd.DataFrame(rows)

styled = (
    df.style
    .hide(axis="index")
    .set_properties(**{"text-align": "center", "font-weight": "bold", "white-space": "pre"})
    .set_table_styles([
        {"selector": "th", "props": [
            ("background-color", "#1d3557"),
            ("color", "white"),
            ("font-weight", "bold"),
            ("text-align", "center"),
            ("padding", "10px 16px"),
            ("font-size", "14px"),
            ("white-space", "normal"),
            ("width", "180px"),
            ("overflow-wrap", "break-word"),
        ]},
        {"selector": "td", "props": [
            ("padding", "10px 20px"),
            ("border", "1px solid #ccc"),
            ("font-size", "14px"),
        ]},
        {"selector": "table", "props": [
            ("border-collapse", "collapse"),
            ("width", "500px"),
            ("table-layout", "fixed"),
        ]},
    ])
)

display(styled)

# Save image 
with open("warming_levels_table.html", "w") as f:
      f.write(styled.to_html())

dfi.export(styled, "warming_levels_table.png",
  table_conversion="matplotlib")

In [ ]:
from holoviews import opts
from bokeh.models import Arrow, NormalHead

gwl_timeidx = pd.read_csv("data/gwl_1850-1900ref_timeidx.csv", index_col="time")
gwl_timeidx.index = pd.to_datetime(gwl_timeidx.index, format="mixed")
annual = gwl_timeidx.resample("YE").mean()
annual.index = annual.index.year
annual = annual[(annual.index >= 1950) & (annual.index <= 2070)]

two_models   = ["EC-Earth3-Veg_r1i1p1f1_ssp585", "MIROC6_r11i1p1f1_ssp585"]
model_labels = ["Model 1", "Model 2"]
model_colors = ["#8B0000", "#003466"]  # dark red for early, blue for late

wl_low, wl_high = 0.8, 2.0
ref_start, ref_end = 1980, 2010
ref_center = (ref_start + ref_end) // 2

ARROW_COLOR = "#E07B39"

def make_panel_hook(arrow_specs):
    def hook(plot, element):
        p = plot.handles['plot']
        for x, y_start, y_end, color in arrow_specs:
            p.add_layout(Arrow(
                end=NormalHead(fill_color=color, line_color=color, size=10),
                x_start=x, y_start=y_start,
                x_end=x, y_end=y_end,
                line_color=color, line_width=1.5,
            ))
    return hook

def make_trajectories(elements):
    for col, label, color in zip(two_models, model_labels, model_colors):
        series = annual[col].dropna()
        df = pd.DataFrame({"Year": series.index, "GWL (°C)": series.values, "Model": label})
        elements.append(
            hv.Curve(df, kdims=["Year"], vdims=["GWL (°C)", "Model"], label=label).opts(
                color=color, alpha=0.8, line_width=2,
            )
        )

def make_scatter(x, y, label, color):
    return hv.Scatter(
        pd.DataFrame({"Year": [x], "GWL (°C)": [y], "Model": [label]}),
        kdims=["Year"], vdims=["GWL (°C)", "Model"],
        label=label,
    ).opts(color=color, size=10, tools=["hover"])

# ---- LEFT PANEL: Consistent Periods ----
left = []
left_arrow_specs = []
make_trajectories(left)
left.append(hv.HLine(wl_low).opts(color="gray", line_dash="dotted", line_width=1.5, tools=[]))
left.append(hv.HLine(wl_high).opts(color="gray", line_dash="dotted", line_width=1.5, tools=[]))

for col, label, color in zip(two_models, model_labels, model_colors):
    series = annual[col].dropna()
    c_low  = series.index[series > wl_low]
    c_high = series.index[series > wl_high]
    if len(c_low) > 0 and len(c_high) > 0:
        yr_low, yr_high = int(c_low[0]), int(c_high[0])
        left.append(make_scatter(yr_low,  wl_low,  label, color))
        left.append(make_scatter(yr_high, wl_high, label, color))
        left_arrow_specs.append((yr_low, wl_low, wl_high, ARROW_COLOR))

left_panel = hv.Overlay(left).opts(
    opts.Overlay(
        title="Consistent Periods",
        xlabel="Year", ylabel="Global Warming Level (°C)",
        ylim=(0, 4), xlim=(1950, 2070),
        width=420, height=370,
        legend_position="top_left",
        hooks=[make_panel_hook(left_arrow_specs)],
    )
)

# ---- RIGHT PANEL: Inconsistent Periods ----
right = []
make_trajectories(right)
right.append(hv.HLine(wl_high).opts(color="gray", line_dash="dotted", line_width=1.5, tools=[]))

teal_label = "Time-based ref. period"
ref_xs, ref_ys = [], []
for col, label, color in zip(two_models, model_labels, model_colors):
    series = annual[col].dropna()
    c_high = series.index[series > wl_high]
    if len(c_high) > 0 and ref_center in series.index:
        yr_high    = int(c_high[0])
        gwl_at_ref = float(series.loc[ref_center])
        ref_xs.append(ref_center)
        ref_ys.append(gwl_at_ref)
        right.append(make_scatter(yr_high, wl_high, label, color))

right_arrow_specs = [(ref_center, min(ref_ys), wl_high, ARROW_COLOR)] if ref_ys else []

if ref_xs:
    right.append(
        hv.Scatter(
            pd.DataFrame({"Year": ref_xs, "GWL (°C)": ref_ys}),
            kdims=["Year"], vdims=["GWL (°C)"],
            label=teal_label,
        ).opts(
            opts.Scatter(
                fill_color=None, line_color="black",
                marker="circle", size=10, line_width=2,
                tools=["hover"],
            )
        )
    )

right_panel = hv.Overlay(right).opts(
    opts.Overlay(
        title="Inconsistent Periods",
        xlabel="Year", ylabel="Global Warming Level (°C)",
        ylim=(0, 4), xlim=(1950, 2070),
        width=420, height=370,
        legend_position="top_left",
        hooks=[make_panel_hook(right_arrow_specs)],
    )
)

fig = (left_panel + right_panel).opts(opts.Layout(shared_axes=False))

caption = (
    "Figure: Consistent vs. Inconsistent GWL Reference Periods. "
    "Each line shows the smoothed global mean surface temperature trajectory for an individual CMIP6 model simulation "
    "relative to 1850–1900. "
    "Left: Using GWL-based reference periods, the change signal between 0.8°C and 2.0°C is a consistent 1.2°C "
    "across all simulations (orange). "
    f"Right: Using a fixed time-based reference period ({ref_start}–{ref_end}, center year {ref_center}), each simulation corresponds to "
    "a different amount of warming at the reference period, introducing additional uncertainty into the change signal (orange)."
)
print(caption)
fig

In [ ]:
with open("gwl_periods.html", "w") as f:
    f.write(file_html(hv.render(fig), CDN))
print("Saved gwl_periods.html")